# Clase 8 · Notebook 01
# La idea adversaria: una GAN que aprende una distribución

**Introducción al Deep Learning — Módulo 3, Unidad 3: Redes Generativas Adversarias (GANs)**

Antes de generar imágenes, entendamos la **mecánica adversaria** con el ejemplo más simple posible:
una GAN que aprende a generar números de una **distribución objetivo** (una campana de Gauss).

- **Generador**: convierte ruido aleatorio en números.
- **Discriminador**: decide si un número es real (de la distribución) o falso (del generador).
- Ambos compiten y mejoran a la vez.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. La distribución real que queremos imitar

Los datos "reales" serán números de una normal centrada en 4. El generador deberá aprender a producir
números parecidos partiendo solo de ruido.

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
tf.random.set_seed(42); np.random.seed(42)

MEDIA, DESV = 4.0, 0.5
def datos_reales(n):
    return np.random.normal(MEDIA, DESV, size=(n, 1)).astype("float32")

plt.figure(figsize=(7, 3))
plt.hist(datos_reales(2000), bins=40, color="#4355FF", alpha=0.7)
plt.title("Distribución real objetivo  N(4, 0.5)"); plt.tight_layout(); plt.show()

## 2. Generador y discriminador

- **Generador**: entra ruido (vector de 8) y sale 1 número.
- **Discriminador**: entra 1 número y sale la probabilidad de que sea real (sigmoide).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

DIM_RUIDO = 8

generador = Sequential([
    Input(shape=(DIM_RUIDO,)),
    Dense(16, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1),                       # salida: un número
], name="generador")

discriminador = Sequential([
    Input(shape=(1,)),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid"), # salida: probabilidad de ser real
], name="discriminador")

bce = tf.keras.losses.BinaryCrossentropy()
opt_g = tf.keras.optimizers.Adam(0.001)
opt_d = tf.keras.optimizers.Adam(0.001)
print("Modelos creados.")

## 3. El bucle de entrenamiento adversario

En cada paso:
1. El **discriminador** aprende con datos reales (etiqueta 1) y falsos (etiqueta 0).
2. El **generador** aprende a que el discriminador clasifique sus números como reales (etiqueta 1).

In [ ]:
def ruido(n):
    return tf.random.normal((n, DIM_RUIDO))

@tf.function
def paso_entrenamiento(reales):
    n = tf.shape(reales)[0]
    # --- Entrenar discriminador ---
    falsos = generador(ruido(n))
    with tf.GradientTape() as td:
        pred_real = discriminador(reales)
        pred_falso = discriminador(falsos)
        loss_d = bce(tf.ones_like(pred_real), pred_real) + bce(tf.zeros_like(pred_falso), pred_falso)
    grad_d = td.gradient(loss_d, discriminador.trainable_variables)
    opt_d.apply_gradients(zip(grad_d, discriminador.trainable_variables))
    # --- Entrenar generador (quiere que sus falsos parezcan reales) ---
    with tf.GradientTape() as tg:
        falsos = generador(ruido(n))
        pred = discriminador(falsos)
        loss_g = bce(tf.ones_like(pred), pred)
    grad_g = tg.gradient(loss_g, generador.trainable_variables)
    opt_g.apply_gradients(zip(grad_g, generador.trainable_variables))
    return loss_d, loss_g

for paso in range(1200):
    ld, lg = paso_entrenamiento(datos_reales(128))
    if paso % 300 == 0:
        print(f"paso {paso:4d} | loss_D {ld.numpy():.3f} | loss_G {lg.numpy():.3f}")
print("Entrenamiento terminado.")

## 4. ¿Ha aprendido el generador la distribución?

Comparamos los números reales con los que genera la red a partir de ruido.

In [ ]:
generados = generador(ruido(2000)).numpy()

plt.figure(figsize=(7, 4))
plt.hist(datos_reales(2000), bins=40, color="#4355FF", alpha=0.6, label="reales")
plt.hist(generados, bins=40, color="#FF647E", alpha=0.6, label="generados (GAN)")
plt.title("Reales vs. generados"); plt.legend(); plt.tight_layout(); plt.show()

print("Media real ~ %.2f | Media generada ~ %.2f" % (MEDIA, generados.mean()))

El histograma de los números generados se aproxima al de los reales: el generador ha aprendido a imitar
la distribución **sin verla nunca directamente**, solo a través del veredicto del discriminador.

## Resumen

- Una GAN entrena **dos redes que compiten**: generador (crea) y discriminador (juzga).
- El generador parte de **ruido** y aprende de la **retroalimentación** del discriminador.
- Tras el entrenamiento, los datos generados se parecen a los reales.

➡️ En el **Notebook 02** aplicaremos esta misma idea para **generar imágenes** de dígitos.